# PTDF bandwidth (`h`) sensitivity sweep

Runs `battery_siting_score.run.main()` 50 times, each time with
`kernel.ptdf_bandwidth` set to a different value and everything else in
`config.yaml` left untouched. Each run's full output CSV is written to its
own subfolder under `out/h_sweep_<timestamp>/`, and a summary table
(`sweep_summary.csv`) collects one row per `h` with headline stats plus a
rank-stability check (Spearman correlation of each run's ranking against the
run closest to the baseline `h = 1.0`).

**Assumptions baked in below (change the config cell if these are wrong):**
- Sweep parameter: `kernel.ptdf_bandwidth` (confirmed to be the only
  continuous "h" hyperparameter active — `kernel.type` is `"ptdf"` in the
  current config, so `kernel.bandwidth_h_m` is unused; `kernel.ptdf_shape` is
  a separate categorical knob left fixed at its configured value).
- Range: 50 points, **log-spaced from 0.1 to 10** (PTDF-distance is
  typically O(0.1)-O(10), so this spans "too peaky" to "too smooth").
- Everything else (paths, wedge settings, window, battery source, run
  method, etc.) is read from the existing `config.yaml` and left as-is.

**Runtime note:** each iteration re-runs the *entire* pipeline (data load,
network build, PyPSA PTDF computation, wedge, scoring) from scratch, exactly
like calling `python run.py` 50 times — it does not cache the PTDF network
across h values. If `run.method: proxy` in your config this is the fast
screening path; if `run.method: lp` this will be much slower per run. If 50
full runs turns out too slow, the PTDF topology/column computation
(everything before `ptdf_kernel_matrix`) does not depend on `h` and could be
cached across iterations — ask if you'd like that optimization added.

In [ ]:
import sys, copy, time, traceback
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

# battery_siting_score is a package inside ALLES_NEU; add ALLES_NEU (the
# package's parent) to sys.path so `import battery_siting_score.run` works,
# exactly like run.py does when launched as a script.
ALLES_NEU_DIR = Path(r"C:\Users\wachs\Documents\TUM\Seminar EM\ALLES_NEU")
if str(ALLES_NEU_DIR) not in sys.path:
    sys.path.insert(0, str(ALLES_NEU_DIR))

from battery_siting_score import run as bsc_run

BASE_CONFIG_PATH = ALLES_NEU_DIR / "battery_siting_score" / "config.yaml"
with open(BASE_CONFIG_PATH, "r", encoding="utf-8") as f:
    base_cfg = yaml.safe_load(f)

print(f"Loaded base config from {BASE_CONFIG_PATH}")
print(f"  kernel.type            = {base_cfg['kernel']['type']}")
print(f"  kernel.ptdf_bandwidth   = {base_cfg['kernel'].get('ptdf_bandwidth')}  (current value, will be swept)")
print(f"  kernel.ptdf_shape       = {base_cfg['kernel'].get('ptdf_shape')}  (left fixed)")
print(f"  run.method              = {base_cfg['run'].get('method')}")
print(f"  paths.out_dir           = {base_cfg['paths']['out_dir']}")

In [ ]:
# --- Sweep configuration: edit these if you want a different range/count ---
H_MIN, H_MAX = 0.1, 10.0
N_H = 50
H_VALUES = np.logspace(np.log10(H_MIN), np.log10(H_MAX), N_H)

BASELINE_H = 1.0  # used only for the rank-stability comparison below

print(f"{N_H} h values, log-spaced from {H_MIN} to {H_MAX}:")
print(np.array2string(H_VALUES, precision=4, separator=", "))

In [ ]:
# --- Set up the sweep's output folder tree ---
base_out_dir = Path(base_cfg["paths"]["out_dir"])
stamp = time.strftime("%Y%m%d_%H%M%S")
sweep_root = base_out_dir / f"h_sweep_{stamp}"
tmp_config_dir = sweep_root / "_tmp_configs"
tmp_config_dir.mkdir(parents=True, exist_ok=True)

print(f"Sweep root: {sweep_root}")

In [ ]:
# --- Run the sweep ---
# Incremental-save note: run.main() itself already writes each iteration's
# full detailed CSV to disk before returning, so per-h results are never
# lost even if a later iteration fails or the kernel dies. This loop ALSO
# rewrites sweep_summary.csv after every single iteration (not just at the
# end), so the aggregated table survives a crash/interrupt partway through
# too -- worst case you lose the in-progress iteration, nothing before it.

summary_rows = []
dV_by_h = {}   # h -> pd.Series indexed by battery_idx, values = dV_proxy_eur_per_year
summary_csv_path = sweep_root / "sweep_summary.csv"

def _save_summary_so_far():
    pd.DataFrame(summary_rows).sort_values("h").reset_index(drop=True).to_csv(
        summary_csv_path, index=False
    )

for i, h in enumerate(H_VALUES):
    run_label = f"h_{i:02d}_{h:.4f}"
    run_out_dir = sweep_root / run_label
    print("=" * 80)
    print(f"[{i+1}/{N_H}] h = {h:.4f}  ->  {run_out_dir}")

    # Deep-copy the base config, override ONLY ptdf_bandwidth and out_dir
    cfg = copy.deepcopy(base_cfg)
    cfg["kernel"]["ptdf_bandwidth"] = float(h)
    cfg["paths"]["out_dir"] = str(run_out_dir)

    tmp_cfg_path = tmp_config_dir / f"{run_label}.yaml"
    with open(tmp_cfg_path, "w", encoding="utf-8") as f:
        yaml.safe_dump(cfg, f, sort_keys=False)

    t0 = time.time()
    try:
        full = bsc_run.main(cfg_path=str(tmp_cfg_path))
        elapsed = time.time() - t0

        sort_col = "dV_proxy_eur_per_year" if "dV_proxy_eur_per_year" in full.columns else "dV_eur_per_year"
        dV = full[sort_col]
        top = full.iloc[0]

        dV_by_h[h] = pd.Series(dV.to_numpy(), index=full["battery_idx"].to_numpy())

        summary_rows.append({
            "h": h,
            "run_label": run_label,
            "status": "ok",
            "elapsed_s": elapsed,
            "n_sites": len(full),
            "mean_dV": dV.mean(),
            "median_dV": dV.median(),
            "std_dV": dV.std(),
            "max_dV": dV.max(),
            "top_battery_idx": int(top["battery_idx"]),
            "top_lat": float(top.get("lat", np.nan)),
            "top_lon": float(top.get("lon", np.nan)),
            "top_dV": float(top[sort_col]),
            "out_dir": str(run_out_dir),
        })
        print(f"  done in {elapsed:.1f}s  mean_dV={dV.mean():,.0f}  top battery_idx={int(top['battery_idx'])}")

    except Exception as e:
        elapsed = time.time() - t0
        print(f"  FAILED after {elapsed:.1f}s: {e}")
        traceback.print_exc()
        summary_rows.append({
            "h": h, "run_label": run_label, "status": f"error: {e}",
            "elapsed_s": elapsed, "n_sites": np.nan, "mean_dV": np.nan,
            "median_dV": np.nan, "std_dV": np.nan, "max_dV": np.nan,
            "top_battery_idx": np.nan, "top_lat": np.nan, "top_lon": np.nan,
            "top_dV": np.nan, "out_dir": str(run_out_dir),
        })

    # Rewrite the summary CSV after EVERY iteration (success or failure) so
    # progress is never lost if the sweep is interrupted or crashes later.
    _save_summary_so_far()

print("=" * 80)
print(f"Sweep finished: {sum(r['status']=='ok' for r in summary_rows)}/{N_H} runs succeeded")
print(f"Summary (kept up to date throughout) -> {summary_csv_path}")

In [ ]:
# --- Rank-stability check: Spearman corr of each run's dV ranking vs the run closest to h=1.0 ---
from scipy.stats import spearmanr

ok_hs = [h for h in H_VALUES if h in dV_by_h]
if ok_hs:
    baseline_h = min(ok_hs, key=lambda h: abs(h - BASELINE_H))
    baseline_series = dV_by_h[baseline_h]
    print(f"Baseline for rank comparison: h = {baseline_h:.4f}")

    spearman_vs_baseline = {}
    for h, series in dV_by_h.items():
        common_idx = series.index.intersection(baseline_series.index)
        rho, _ = spearmanr(series.loc[common_idx], baseline_series.loc[common_idx])
        spearman_vs_baseline[h] = rho
else:
    spearman_vs_baseline = {}
    print("No successful runs -- skipping rank-stability check")

summary_df = pd.DataFrame(summary_rows).sort_values("h").reset_index(drop=True)
summary_df["spearman_vs_baseline_h"] = summary_df["h"].map(spearman_vs_baseline)

# Final overwrite: same path used for the per-iteration incremental saves above,
# now with the spearman column added.
summary_df.to_csv(summary_csv_path, index=False)
print(f"Summary saved -> {summary_csv_path}")
summary_df

In [ ]:
# --- Quick plot: mean dV and rank stability vs h ---
import matplotlib.pyplot as plt

ok = summary_df[summary_df["status"] == "ok"]

fig, axes = plt.subplots(2, 1, figsize=(8, 8), sharex=True)

axes[0].plot(ok["h"], ok["mean_dV"], marker="o")
axes[0].set_xscale("log")
axes[0].set_ylabel("mean dV_proxy (EUR/yr)")
axes[0].set_title("Mean locational value vs PTDF bandwidth h")
axes[0].axvline(BASELINE_H, color="gray", linestyle="--", linewidth=1, label=f"h={BASELINE_H} (config default)")
axes[0].legend()

axes[1].plot(ok["h"], ok["spearman_vs_baseline_h"], marker="o", color="darkorange")
axes[1].set_xscale("log")
axes[1].set_xlabel("h (ptdf_bandwidth, log scale)")
axes[1].set_ylabel(f"Spearman rho vs h={BASELINE_H}")
axes[1].set_title("Ranking stability vs PTDF bandwidth h")
axes[1].axvline(BASELINE_H, color="gray", linestyle="--", linewidth=1)

plt.tight_layout()
fig_path = sweep_root / "h_sweep_summary.png"
plt.savefig(fig_path, dpi=150)
print(f"Plot saved -> {fig_path}")
plt.show()

## Interactive map: how the best sites shift with h

Builds one self-contained HTML page with a slider that scrubs across every
successful run in this sweep. Dragging the slider re-colors the (fixed) grid
of candidate cells and moves the top-10 markers to match whichever `h` is
selected -- this is the direct visual answer to "how does the best-site
ranking change with the PTDF bandwidth". A small sparkline (mean dV in grey,
top-site dV in orange, both vs h) sits above the map and can be clicked to
jump the slider straight to that run.

Uses `battery_siting_score.plot_grid_map_html.build_sweep_html`, added
alongside the existing static single-run map (`build_html`, used by
`plot_grid_map_html.py`'s normal CLI) -- it does not touch or replace that
code path.

In [ ]:
from battery_siting_score.plot_grid_map_html import build_sweep_html
from IPython.display import IFrame

sweep_map_path = build_sweep_html(sweep_root, baseline_param=BASELINE_H)
print(f"Interactive sweep map -> {sweep_map_path}")
print("Double-click the .html file to open it full-size in a browser, or preview inline below:")

IFrame(src=str(sweep_map_path), width="100%", height=650)